# 02 · Traditional baselines (E1, part of E4)

**Runtime:** CPU (standard). No GPU. **Run after:** 01. **Time:** ~20–40 min.

* Majority class (lower bound)
* TF-IDF + logistic regression with gold evidence (C tuned on validation)
* Extra cheap E4 points: TF-IDF claim-only and TF-IDF with retrieved evidence

In [ ]:
# Setup: Drive, code, packages, paths  (expected runtime: CPU)
from google.colab import drive
drive.mount('/content/drive')
import os, sys
BASE = '/content/drive/MyDrive/neural-evidence'
CODE = '/content/drive/MyDrive/Colab Notebooks/neural-evidence'   # your uploaded code folder
REPO_URL = ''
if not os.path.exists(f'{CODE}/ne') and REPO_URL:
    os.system(f'git clone -q {REPO_URL} /content/code'); CODE = '/content/code'
assert os.path.exists(f'{CODE}/ne'), f'Code not found at {CODE}'
os.chdir(CODE); sys.path.insert(0, CODE)
!pip -q install -r requirements.txt 
PROC, RESULTS, CACHE = f'{BASE}/data/processed', f'{BASE}/results', f'{BASE}/cache'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
from ne import runtime
from ne.data import read_jsonl, write_jsonl, eval_sets
runtime.setup()
if runtime.gpu_kind() not in 'CPU'.split('/') and 'CPU' != 'CPU':
    print('WARNING: expected CPU; this runtime is', runtime.gpu_kind(), '- it will still work, just at a different speed/cost')
EVALS = eval_sets(PROC); print('eval sets:', list(EVALS))

In [ ]:
from ne import baselines, results
train, val = read_jsonl(f'{PROC}/fever/train.jsonl'), read_jsonl(f'{PROC}/fever/val.jsonl')
tests = {k: read_jsonl(EVALS[k]) for k in ('fever_test', 'scifact_test')}
labels = {k: [r['label'] for r in v] for k, v in tests.items()}
print(len(train), len(val), {k: len(v) for k, v in tests.items()})

In [ ]:
# Majority class
if not results.is_done(RESULTS, 'E1', 'majority', 0):
    maj = baselines.majority(train)
    results.save_run(RESULTS, 'E1', 'majority', 0, {'predicts': maj},
                     {k: [maj] * len(v) for k, v in tests.items()}, labels, rows_by_set=tests)

In [ ]:
# TF-IDF + LR: gold evidence (E1), claim only and retrieved evidence (E4)
import time
for exp, name, ev in [('E1', 'tfidf_lr', 'gold'), ('E4', 'tfidf_lr_claim_only', 'none'), ('E4', 'tfidf_lr_retrieved', 'retrieved')]:
    if results.is_done(RESULTS, exp, name, 0):
        print('skip', name); continue
    print(f'== {name} (evidence={ev})'); t = time.time()
    model, scores = baselines.tune_tfidf(train, val, ev)
    results.save_run(RESULTS, exp, name, 0, {'evidence': ev, 'val_macro_f1_by_C': scores, 'C': model.C},
                     {k: model.predict(v) for k, v in tests.items()}, labels,
                     extra={'train_seconds': round(time.time() - t, 1)}, rows_by_set=tests)